In [8]:
import duckdb
duckdb.sql("""
    SELECT date, COUNT(*) AS rows, MIN(fetched_at), MAX(fetched_at)
    FROM read_parquet('../data/raw/vehicles/*/*.parquet', hive_partitioning=true)
    GROUP BY date
""").show()

┌────────────┬─────────┬───────────────────────────────┬───────────────────────────────┐
│    date    │  rows   │        min(fetched_at)        │        max(fetched_at)        │
│    date    │  int64  │   timestamp with time zone    │   timestamp with time zone    │
├────────────┼─────────┼───────────────────────────────┼───────────────────────────────┤
│ 2026-10-18 │ 1962660 │ 2026-10-17 23:59:24.118854-04 │ 2026-10-18 16:57:55.078102-04 │
│ 2026-10-17 │   96642 │ 2026-10-17 23:08:51.239581-04 │ 2026-10-17 23:58:53.597428-04 │
└────────────┴─────────┴───────────────────────────────┴───────────────────────────────┘



In [6]:
duckdb.sql("""
    SELECT DISTINCT
        trip_id, route_id, vehicle_id,
        to_timestamp(vehicle_timestamp) AT TIME ZONE 'America/Toronto' AS seen_at
    FROM read_parquet('../data/raw/vehicles/*/*.parquet', hive_partitioning=true)
    WHERE trip_id <> ''
      AND route_id = '29'
      AND to_timestamp(vehicle_timestamp) AT TIME ZONE 'America/Toronto'
          BETWEEN TIMESTAMP '2026-10-18 00:10' AND TIMESTAMP '2026-10-18 09:15'
    ORDER BY seen_at
""").show()

┌───────────┬──────────┬────────────┬─────────────────────┐
│  trip_id  │ route_id │ vehicle_id │       seen_at       │
│  varchar  │ varchar  │  varchar   │      timestamp      │
├───────────┼──────────┼────────────┼─────────────────────┤
│ 10447070  │ 29       │ 9435       │ 2026-10-18 00:10:01 │
│ 11913070  │ 29       │ 9443       │ 2026-10-18 00:10:03 │
│ 126731070 │ 29       │ 9466       │ 2026-10-18 00:10:05 │
│ 131309070 │ 29       │ 9428       │ 2026-10-18 00:10:05 │
│ 29048070  │ 29       │ 9427       │ 2026-10-18 00:10:06 │
│ 115798070 │ 29       │ 9432       │ 2026-10-18 00:10:07 │
│ 117734070 │ 29       │ 9456       │ 2026-10-18 00:10:17 │
│ 10447070  │ 29       │ 9435       │ 2026-10-18 00:10:26 │
│ 115798070 │ 29       │ 9432       │ 2026-10-18 00:10:27 │
│ 93302070  │ 29       │ 9409       │ 2026-10-18 00:10:28 │
│    ·      │ ·        │  ·         │          ·          │
│    ·      │ ·        │  ·         │          ·          │
│    ·      │ ·        │  ·         │   

In [7]:
duckdb.sql("""
    SELECT route_id, COUNT(DISTINCT vehicle_id) AS vehicles
    FROM read_parquet('../data/raw/vehicles/*/*.parquet', hive_partitioning=true)
    WHERE route_id IN ('504', '25', '36', '939')
    GROUP BY route_id
""").show()

┌──────────┬──────────┐
│ route_id │ vehicles │
│ varchar  │  int64   │
├──────────┼──────────┤
│ 504      │       45 │
│ 939      │       15 │
│ 36       │       17 │
│ 25       │       35 │
└──────────┴──────────┘

